<a href="https://colab.research.google.com/github/baharerturul/T-SQL_Practices/blob/main/Colab'e_ho%C5%9F_geldiniz_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Creating a Full Analysis Report**

*   Read data from different sources in Spark
*   Perform SQL operations on a Spark DataFrame
*   Generate statistical measurements in a consistent way
*   Generate graphs and plots using Plotly
*   Compile an analysis report incorporating all the previous steps and data

 Python'da
Spark kullanırken, düz dosyalar, JSON formatındaki REST API'ler ve benzeri çeşitli kaynaklardan
veri okumak gerekebilir. Bu bölüm, farklı veri türlerini okumak, bunlar üzerinde SQL işlemleri gerçekleştirmek, tanımlayıcı
istatistiksel analiz yapmak ve tam bir analiz raporu oluşturmak için hazırlanmıştır.


**Spark'ta Farklı Veri Kaynaklarından Veri Okuma**

Spark'ın avantajlarından biri, çeşitli veri kaynaklarından veri okuyabilme . Ancak bu durum tutarlı değildir ve her Spark sürümüyle birlikte değişmektedir. Bu bölümün bu kısmında CSV ve JSON dosyalarının nasıl okunacağı açıklanacaktır.

**Alıştırma 47: PySpark Nesnesi Kullanarak CSV Dosyasından Veri Okuma**

CSV verilerini okumak için `spark.read.csv("dosya adı .csv uzantısıyla birlikte")` yazmanız gerekir.
Kitapta örnek olarak kullanılan banka verileri kullanılmıştır.



 bank.csv dosyasından verileri okumak için aşağıdaki adımlar sırasıyla uygulanır
*   Öncelikle, gerekli paketleri Jupyter notebook'a aktaralım:






In [18]:
!pip uninstall -y pandas-profiling
!pip install ydata-profiling

import os
import pandas as pd
import numpy as np
import collections
from sklearn.base import TransformerMixin
import random
from ydata_profiling import ProfileReport

2. Ardından, gösterildiği gibi gerekli tüm kütüphaneleri içe aktarılır:

In [2]:
import seaborn as sns
import time
import re
import os
import matplotlib.pyplot as plt



*  Veri setimizi daha görünür hale getirecek ve daha yüksek kontrast sağlayacak olan tick işareti temalarını kullanalım :





In [3]:
sns.set(style="ticks")



*   Spark oturumunu oluşturmak için Spark için gerekli kütüphaneleri içe aktarılır:




In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('ml-bank').getOrCreate()



*    Aşağıdaki komutu kullanarak df_csv Spark nesnesini oluşturduktan sonra CSV veriler okunur:




In [5]:
df_csv = spark.read.csv('bank.csv', sep=';', header = True, inferSchema = True)



*   Aşağıdaki komutu kullanarak şemayı yazdılır ve çıktı oluşturulur:




In [6]:
df_csv.printSchema()


root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



**PySpark Nesnesi Kullanarak JSON Verilerini Okuma**

JSON verilerini okumak için, read.json("dosya adı .json") yazılması gerekilmektedir. Aşağıda dummy bir Json veriseti oluşturulmuştur.


In [7]:
# Temporarily create a dummy bank.json file to proceed with the notebook
# You should upload your actual bank.json file if you have one.
import json

dummy_data = [
    {"id": 1, "name": "Alice", "age": 30},
    {"id": 2, "name": "Bob", "age": 24},
    {"id": 3, "name": "Charlie", "age": 35}
]

with open('bank.json', 'w') as f:
    for item in dummy_data:
        json.dump(item, f)
        f.write('\n') # Spark's JSON reader often expects one JSON object per line

print("Dummy bank.json created.")

Dummy bank.json created.


In [8]:
df_json = spark.read.json('bank.json')

**Spark DataFrame üzerinde SQL işlemleri**


Spark, DataFrame'i sütunlara ayırarak veri üzerindeki sorguları optimize eder; bu da Spark'ın şemayı
anlamasına yardımcı olur. En sık kullanılan SQL işlemlerinden bazıları arasında veriyi alt kümelere ayırma, verileri
birleştirme, filtreleme, belirli sütunları seçme, sütunları silme, tüm boş değerleri silme ve yeni sütunlar
ekleme yer almaktadır.

**Alıştırma 48: PySpark'ta Veri Okuma ve SQL İşlemleri Gerçekleştirme**

Verilerin özet istatistikleri için, DataFrame'deki tüm sütunlar için sayım, ortalama, standart sapma, maksimum
ve minimum değerler hakkında bilgi sağlayacak olan spark_df.**describe**().show() fonksiyonunu kullanabiliriz .
Kitapta yer alan örnek banka veri setinde özet istatistik verileri aşağıdaki şekilde elde edilebilir:


*  Yeni bir Jupyter not defteri oluşturduktan sonra, gerekli tüm paketleri içe
aktarın:





In [9]:
import os
import pandas as pd
import numpy as np

. Spark oturumunu oluşturmak için Spark için gerekli tüm kütüphaneleri içe aktarılır:

In [10]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('ml-bank').getOrCreate()

 Aşağıdaki şekilde gösterildiği gibi, Spark nesnesini kullanarak CSV dosyasından veri oluşturulup ve okunur:


In [11]:
spark_df = spark.read.csv('bank.csv', sep=';', header = True, inferSchema = True)


 Aşağıdaki komutu kullanarak Spark nesnesinden ilk beş satırı yazdırılır:

In [12]:
spark_df.head(5)


[Row(age=30, job='unemployed', marital='married', education='primary', default='no', balance=1787, housing='no', loan='no', contact='cellular', day=19, month='oct', duration=79, campaign=1, pdays=-1, previous=0, poutcome='unknown', y='no'),
 Row(age=33, job='services', marital='married', education='secondary', default='no', balance=4789, housing='yes', loan='yes', contact='cellular', day=11, month='may', duration=220, campaign=1, pdays=339, previous=4, poutcome='failure', y='no'),
 Row(age=35, job='management', marital='single', education='tertiary', default='no', balance=1350, housing='yes', loan='no', contact='cellular', day=16, month='apr', duration=185, campaign=1, pdays=330, previous=1, poutcome='failure', y='no'),
 Row(age=30, job='management', marital='married', education='tertiary', default='no', balance=1476, housing='yes', loan='yes', contact='unknown', day=3, month='jun', duration=199, campaign=4, pdays=-1, previous=0, poutcome='unknown', y='no'),
 Row(age=59, job='blue-coll

Yapılandırılmış veriye geçmek için öncelikle veri türlerini belirleyelim. Her sütunun veri türünü
yazdırmak için aşağıdaki komutu kullanın:


In [13]:
spark_df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



 Şimdi, isim içeren satır ve sütunların toplam sayısını hesaplanır:

In [14]:
spark_df.count()

4521

In [15]:
len(spark_df.columns), spark_df.columns

(17,
 ['age',
  'job',
  'marital',
  'education',
  'default',
  'balance',
  'housing',
  'loan',
  'contact',
  'day',
  'month',
  'duration',
  'campaign',
  'pdays',
  'previous',
  'poutcome',
  'y'])

Aşağıdaki komutu kullanarak DataFrame'in özet istatistiklerini yazdırılır:


In [16]:
spark_df.describe().show()

+-------+------------------+-------+--------+---------+-------+------------------+-------+----+--------+------------------+-----+------------------+------------------+------------------+------------------+--------+----+
|summary|               age|    job| marital|education|default|           balance|housing|loan| contact|               day|month|          duration|          campaign|             pdays|          previous|poutcome|   y|
+-------+------------------+-------+--------+---------+-------+------------------+-------+----+--------+------------------+-----+------------------+------------------+------------------+------------------+--------+----+
|  count|              4521|   4521|    4521|     4521|   4521|              4521|   4521|4521|    4521|              4521| 4521|              4521|              4521|              4521|              4521|    4521|4521|
|   mean| 41.17009511170095|   NULL|    NULL|     NULL|   NULL|1422.6578190665782|   NULL|NULL|    NULL|15.9152842291528

Bir DataFrame'den birden fazla sütun seçmek için `spark_df.select('col1', 'col2', 'col3')` fonksiyonunu
kullanabiliriz . Örneğin, `balance` sütunundan ilk beş satırı seçelim.


In [19]:
spark_df.select('""balance""','""y"""').show(5)

{"ts": "2025-12-24 08:09:30.067", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `\"\"balance\"\"` cannot be resolved. Did you mean one of the following? [`balance`, `age`, `contact`, `loan`, `campaign`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o48.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `\"\"balance\"\"` cannot be resolved. Did you mean one of the following? [`balance`, `age`, `contact`, `loan`, `campaign`]. SQLSTATE: 42703;\n'Project ['\"\"balance\"\", '\"\"y\"\"\"]\n+- Relation [age#60,job#61,marital#62,education#63,default#64,balance#65,hou

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `""balance""` cannot be resolved. Did you mean one of the following? [`balance`, `age`, `contact`, `loan`, `campaign`]. SQLSTATE: 42703;
'Project ['""balance"", '""y"""]
+- Relation [age#60,job#61,marital#62,education#63,default#64,balance#65,housing#66,loan#67,contact#68,day#69,month#70,duration#71,campaign#72,pdays#73,previous#74,poutcome#75,y#76] csv


İki değişken arasındaki ilişkiyi, seviyelerinin sıklığı açısından belirlemek için çapraz tablo kullanılabilir. İki
sütun arasında çapraz tablo elde etmek için spark_df.crosstab('col1', col2) fonksiyonunu
kullanabiliriz. Çapraz tablo, sayısal değişkenler arasında değil, iki kategorik değişken arasında
gerçekleştirilir:

In [ ]:
spark_df.crosstab('""y"""', '""marital""').show()

 Şimdi veri setine yeni bir sütun ekleyelim:


In [ ]:
from pyspark.sql.functions import col, regexp_replace, when, lit
from pyspark.sql.types import DoubleType

# sample sets
sample1 = spark_df.sample(False, 0.2, 42)
sample2 = spark_df.sample(False, 0.2, 43)
# train set
train = spark_df.sample(False, 0.8, 44)

# Correcting the column name, cleaning it from quotes, and handling non-numeric values for casting
train_df_with_balance_new = (
    train.withColumn('cleaned_balance_str', regexp_replace(col('""balance""'), '""', ''))
    .withColumn('balance_new',
                when(col('cleaned_balance_str').isin('no', 'yes'), lit(None).cast(DoubleType()))
                .otherwise(col('cleaned_balance_str').cast(DoubleType())) / 2.0)
)
train_df_with_balance_new.select('""balance""', 'balance_new').show(5)

 Yeni oluşturulan sütunu aşağıdaki komutu kullanarak silinebilinir:



In [ ]:
train.drop('balance_new')


**İki DataFrame Oluşturma ve Birleştirme**

Amaç, PySpark kullanarak bir
Spark DataFrame üzerinde birleştirme işlemleri gerçekleştirmektir.
Veriler, Portekizli bir bankacılık kurumunun doğrudan pazarlama kampanyalarıyla ilgilidir. Mevcut banka pazarlama verilerinden iki adet DataFrame oluşturup ve bunları birincil anahtar
temelinde birleştirelim

. Öncelikle, gerekli başlık dosyalarını Jupyter notebook'a aktaralım:

In [ ]:
import os
import pandas as pd
import numpy as np
import pyspark

 Spark oturumunu oluşturmak için Spark için gerekli tüm kütüphaneleri içe aktarılır:

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('ml-bank').getOrCreate()

 Aşağıdaki komutu kullanarak CSV dosyalarındaki verileri bir Spark nesnesine okunur:


In [ ]:
spark_df = spark.read.csv('bank.csv', sep=';', header = True, inferSchema = True)

 Spark nesnesinden ilk beş satırı yazdırın:


In [ ]:
spark_df.head(5)


İki DataFrame'i birincil anahtar (ID) kullanarak birleştirmek için öncelikle yapılacak adımlar şu şekildedir:
Öncelikle, ID sütunu içeren yeni bir DataFrame eklenecektir :


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id
train_with_id = spark_df.withColumn("ID", monotonically_increasing_id())

Ardından, ID2 adında başka bir sütun oluşturulur:

In [ ]:
train_with_id = train_with_id.withColumn('ID2', train_with_id.ID)

 Aşağıdaki komutu kullanarak DataFrame'i bölünür:

In [ ]:
train_with_id1 = train_with_id.drop('balance', "ID2")
train_with_id2 = train_with_id.select('balance', "ID2")

train_with_id2 tablosunun ID sütun adlarını değiştirilir :


In [ ]:
train_with_id2 = train_with_id2.withColumnRenamed("ID2", "ID")

Aşağıdaki komutu kullanarak train_with_id1 ve train_with_id2'yi birleştirilir:


In [ ]:
train_merged = train_with_id1.join(train_with_id2, on=['ID'], how='left_outer')

**DataFrame Alt Kümelere Ayrılması**

Amaç, PySpark kullanarak Spark DataFrame üzerinde filtreleme/alt kümeleme işlemleri gerçekleştirmektir.

Banka pazarlama verilerinde bakiyenin 0'dan büyük olduğu verileri DataFrame'den alt küme olarak alalım :

 DataFrame'i alt kümelere ayırmak ve filtrelemek için SQL sorguları çalıştırılır:

In [ ]:
train_subsetted = spark_df.filter(spark_df.balance > 0.0)
pd.DataFrame(train_subsetted.head(5))

**İstatistiksel Ölçümlerin Oluşturulması**

Python, istatistiksel modüllere sahip genel amaçlı bir dildir. Sayısal değişkenler için veri
dağılımını belirleme, korelasyon matrisi oluşturma, kategorik değişkenlerdeki seviyelerin
sıklığını ve modunu belirleme gibi birçok istatistiksel analiz Python ile
gerçekleştirilebilir.

**Plotly Kullanarak Görselleştirme Oluşturma**

Plotly'nin Python grafik kütüphanesi, etkileşimli ve yayın kalitesinde grafikler oluşturmanızı sağlar.

Bu başlıkta, UCI Makine Öğrenimi Veri Deposundan banka pazarlama verilerini çıkarıp
kullanılacaktır. Amaç, Python'da Plotly kullanarak görselleştirmeler oluşturmaktır.



 Veri görselleştirmeyi Plotly ile gerçekleştirmek için gerekli kütüphaneleri içe aktarılır:

In [ ]:
!pip install chart-studio
import plotly.graph_objects as go
from chart_studio.plotly import iplot
import plotly as py

In [ ]:
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

Plotly kullanarak grafikleri çizmek için gerekli kütüphaneleri içe aktarılır:

In [20]:
import pandas as pd

# Dosyayı oku
df = pd.read_csv("bank.csv", sep=';')
# df = pd.read_csv("bank-full.csv")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [ ]:
pip uninstall chart-studio


In [21]:
import plotly.express as px

fig = px.histogram(
    df,
    x="balance",
    nbins=50,
    title="Balance Dağılımı",
    labels={"balance": "Bakiye"}
)

fig.show()

In [22]:
import plotly.express as px

fig_density = px.histogram(
    df,
    x="balance",
    nbins=50,
    histnorm='density', # This makes it a density plot
    title="Balance Yoğunluk Grafiği",
    labels={"balance": "Bakiye"}
)

fig_density.show()

In [23]:
fig = px.box(
    df,
    x="job",
    y="balance",
    title="Mesleğe Göre Balance Dağılımı"
)

fig.show()


In [24]:
housing_counts = df["housing"].value_counts().reset_index()
housing_counts.columns = ["housing", "count"]

fig = px.bar(
    housing_counts,
    x="housing",
    y="count",
    title="Housing (Konut Kredisi) Bar Grafiği"
)

fig.show()


In [ ]:
import plotly.express as px

fig_box = px.box(df, y="balance", title="Balance Dağılımı (Kutu Grafiği)", labels={"balance": "Bakiye"})
fig_box.show()

**SONUÇ**


Bu bölümde, çeşitli kaynaklardan verileri Spark ortamına Spark DataFrame olarak nasıl
aktaracağımızı öğrendik. Ayrıca, bu DataFrame üzerinde çeşitli SQL işlemlerini nasıl
gerçekleştireceğimizi ve korelasyon analizi, veri dağılımının belirlenmesi, özellik önem modelinin
oluşturulması gibi çeşitli istatistiksel ölçümlerin nasıl üretileceğini öğrendik. Ayrıca, analiz
raporu geliştirmek için çeşitli grafikler oluşturabileceğiniz Plotly kütüphanesini kullanarak etkili
grafikler oluşturmayı da incelemiş olduk.
